#Severity Estimation Model

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# 1. Load Data
df = pd.read_csv('synthetic_crime_severity_real_zones.csv')

# 2. Create Classes (Low/Medium/High) from the Score
# This allows us to calculate F1 Score and Accuracy
bins = [-1, 33, 66, 101]
labels = ['Low', 'Medium', 'High']
df['Severity_Class'] = pd.cut(df['Harm_Score'], bins=bins, labels=labels)

# 3. Preprocessing
le_enc = LabelEncoder()
df['crime_code'] = le_enc.fit_transform(df['crime'])
df['zone_code'] = le_enc.fit_transform(df['Patrol_Zone_DS'])
df['Hour'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.hour.fillna(0)

X = df[['crime_code', 'Hour', 'zone_code', 'victim_age']]
y = le_enc.fit_transform(df['Severity_Class']) # Encode Target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 4. Compare Models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier()
}

print(f"{'Model':<20} | {'Accuracy':<10} | {'F1 Score':<10}")
print("-" * 45)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{name:<20} | {acc:.4f}     | {f1:.4f}")

Model                | Accuracy   | F1 Score  
---------------------------------------------
Random Forest        | 0.9482     | 0.9483
Gradient Boosting    | 0.9503     | 0.9504
SVM                  | 0.8571     | 0.8538
KNN                  | 0.7598     | 0.7616


mock data for hotspot prediction

In [5]:
import numpy as np
import pandas as pd

# ---- Load your GN list ----
gn_path = "gn_with_closest_police_station.csv"
gn_df = pd.read_csv(gn_path)

gn_df = gn_df.rename(columns={"admin4Pcode": "gn_code", "admin4Name_en": "gn_english_name"})

required_cols = ["gn_code", "GN_population", "distance_to_station_km"]
missing = [c for c in required_cols if c not in gn_df.columns]
if missing:
    raise ValueError(f"Missing columns in GN file: {missing}")

gn_df = gn_df.dropna(subset=["gn_code"]).copy()
gn_df = gn_df.sort_values("gn_code").reset_index(drop=True)

# Create gn_encoded
gn_df["gn_encoded"] = np.arange(len(gn_df), dtype=int)

# ---- Crimes list ----
crimes = ["drugs", "robbery", "theft", "vehical theft", "buglary", "stabbing"]

# ---- Build a realistic base risk signal from real GN features ----
pop = gn_df["GN_population"].fillna(gn_df["GN_population"].median()).astype(float)
dist = gn_df["distance_to_station_km"].fillna(gn_df["distance_to_station_km"].median()).astype(float)

# Normalize helpers
def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

pop_z = zscore(np.log1p(pop))
dist_z = zscore(dist)

base = 0.65 * pop_z + 0.35 * dist_z  # weighted mix

# ---- Crime-specific profiles  ----
crime_profile = {
    "drugs":         {"severity": 1.15, "noise": 0.18},
    "robbery":       {"severity": 1.35, "noise": 0.20},
    "theft":         {"severity": 1.05, "noise": 0.16},
    "vehical theft": {"severity": 1.25, "noise": 0.19},
    "buglary":       {"severity": 1.30, "noise": 0.20},
    "stabbing":      {"severity": 1.45, "noise": 0.22},
}

rng = np.random.default_rng(42)

rows = []
for crime in crimes:
    sev = crime_profile[crime]["severity"]
    noise = crime_profile[crime]["noise"]

    # Create crime-specific signal:
    signal = sev * base + rng.normal(0, noise, size=len(gn_df))

    # Squash into 0..1 like a probability-ish risk score
    risk = 1 / (1 + np.exp(-signal))

    risk = (risk - risk.min()) / (risk.max() - risk.min() + 1e-9)
    risk = np.clip(risk, 0, 1)

    # Build rows
    for i in range(len(gn_df)):
        rows.append({
            "crime_type": crime,
            "gn_encoded": int(gn_df.loc[i, "gn_encoded"]),
            "gn_name": str(gn_df.loc[i, "gn_code"]),
            "risk_score": float(risk[i])
        })

hotspot_predictions = pd.DataFrame(rows)

# ---- Save ----
out_path = "hotspot_predictions.csv"
hotspot_predictions.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Shape:", hotspot_predictions.shape)
print(hotspot_predictions.head(10))


Saved: hotspot_predictions.csv
Shape: (7122, 4)
  crime_type  gn_encoded    gn_name  risk_score
0      drugs           0  LK2103005    0.600258
1      drugs           1  LK2103010    0.579689
2      drugs           2  LK2103015    0.490813
3      drugs           3  LK2103020    0.693728
4      drugs           4  LK2103025    0.109367
5      drugs           5  LK2103030    0.814509
6      drugs           6  LK2103035    0.110677
7      drugs           7  LK2103040    0.379722
8      drugs           8  LK2103045    0.557793
9      drugs           9  LK2103050    0.372394


##Model Training

In [16]:
!pip -q install xgboost

In [17]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# ----------------------------
# 1) Load GN ↔ closest station data
# ----------------------------
gn_path = "gn_with_closest_police_station.csv"
gn_station = pd.read_csv(gn_path)

# Standardize GN identifiers
gn_station = gn_station.rename(columns={
    "admin4Name_en": "gn_name_en",
    "admin4Pcode": "gn_name"
})

required_gn_cols = ["gn_name", "GN_population", "distance_to_station_km", "closest_police_station"]
missing = [c for c in required_gn_cols if c not in gn_station.columns]
if missing:
    raise ValueError(f"GN file missing required columns: {missing}")

gn_station = gn_station[required_gn_cols].copy()

# Clean numeric GN features
gn_station["GN_population"] = pd.to_numeric(gn_station["GN_population"], errors="coerce")
gn_station["distance_to_station_km"] = pd.to_numeric(gn_station["distance_to_station_km"], errors="coerce")

# Fix invalid values
pop_med = gn_station["GN_population"].median()
dist_med = gn_station["distance_to_station_km"].median()

gn_station.loc[gn_station["GN_population"].isna() | (gn_station["GN_population"] <= 0), "GN_population"] = pop_med
gn_station.loc[gn_station["distance_to_station_km"].isna() | (gn_station["distance_to_station_km"] < 0), "distance_to_station_km"] = dist_med

# ----------------------------
# 2) Load HOTSPOT predictions CSV (your real output flattened)
# ----------------------------
hotspot = pd.read_csv("hotspot_predictions.csv")

required_hotspot_cols = ["crime_type", "gn_name", "risk_score"]
missing = [c for c in required_hotspot_cols if c not in hotspot.columns]
if missing:
    raise ValueError(f"Hotspot CSV missing required columns: {missing}")

hotspot["risk_score"] = pd.to_numeric(hotspot["risk_score"], errors="coerce")
hotspot.loc[~hotspot["risk_score"].between(0, 1), "risk_score"] = np.nan
hotspot["risk_score"] = hotspot["risk_score"].fillna(hotspot["risk_score"].median())

# ----------------------------
# 3) Merge hotspot + GN features
# ----------------------------
df = hotspot.merge(gn_station, on="gn_name", how="left")

df["GN_population"] = df["GN_population"].fillna(pop_med)
df["distance_to_station_km"] = df["distance_to_station_km"].fillna(dist_med)
df["closest_police_station"] = df["closest_police_station"].fillna("Unknown")

# ----------------------------
# 4) Feature engineering
# ----------------------------
df["crime_type_enc"] = df["crime_type"].astype("category").cat.codes
df["risk_rank"] = df.groupby("crime_type")["risk_score"].rank(ascending=False, method="first")

# ----------------------------
# 5) Create synthetic allocation_score labels (SAFE)
# ----------------------------
severity_weights = {
    "drugs": 1.2,
    "robbery": 1.5,
    "theft": 1.0,
    "vehical theft": 1.25,
    "buglary": 1.3,
    "stabbing": 1.7
}
df["severity"] = df["crime_type"].map(severity_weights).fillna(1.0)

alpha = 0.03

df["raw_allocation"] = (
    df["risk_score"]
    * df["severity"]
    * (1 + alpha * df["distance_to_station_km"])
    * np.log1p(df["GN_population"])
)

df["raw_allocation"] = df["raw_allocation"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

def safe_normalize(x: pd.Series) -> pd.Series:
    s = float(x.sum())
    if not np.isfinite(s) or s <= 0:
        return pd.Series(np.full(len(x), 1.0 / len(x)), index=x.index)
    return x / s

df["allocation_score"] = df.groupby("crime_type")["raw_allocation"].transform(safe_normalize)

# Final label checks
df["allocation_score"] = pd.to_numeric(df["allocation_score"], errors="coerce")
df["allocation_score"] = df["allocation_score"].replace([np.inf, -np.inf], np.nan)

df["allocation_score"] = df.groupby("crime_type")["allocation_score"].transform(
    lambda x: x.fillna(1.0 / len(x))
)

assert np.isfinite(df["allocation_score"]).all(), "allocation_score still has NaN/INF"

print("✅ Label OK | min/max:", df["allocation_score"].min(), df["allocation_score"].max())

# ----------------------------
# 6) Train XGBoost
# ----------------------------
features = ["risk_score", "distance_to_station_km", "GN_population", "crime_type_enc", "risk_rank"]
X = df[features]
y = df["allocation_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBRegressor(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_test)
print("✅ Model trained | MAE:", mean_absolute_error(y_test, pred))

# ----------------------------
# 7) Allocate officers
# ----------------------------
crime_of_interest = "drugs"     # change as needed
total_officers = 500
max_gns_to_cover = 80           # only deploy to top K GNs
min_per_gn = 1                  # minimum only applies inside the top-K set

sub = df[df["crime_type"] == crime_of_interest].copy()

# Predict allocation weights
sub["pred_alloc"] = model.predict(sub[features])
sub["pred_alloc"] = sub["pred_alloc"].clip(lower=0)

# Select TOP-K by predicted allocation
sub = sub.sort_values("pred_alloc", ascending=False).reset_index(drop=True)

deploy = sub.head(max_gns_to_cover).copy()
non_deploy = sub.iloc[max_gns_to_cover:].copy()

# Normalize weights within deploy set
s = deploy["pred_alloc"].sum()
if s <= 0 or not np.isfinite(s):
    deploy["pred_alloc_norm"] = 1.0 / len(deploy)
else:
    deploy["pred_alloc_norm"] = deploy["pred_alloc"] / s

# Allocate officers to deploy set
deploy["assigned_officers"] = np.floor(deploy["pred_alloc_norm"] * total_officers).astype(int)

# Enforce minimum within deploy set
deploy.loc[deploy["assigned_officers"] < min_per_gn, "assigned_officers"] = min_per_gn

# Fix total to match exactly (after min constraint)
diff = int(total_officers - deploy["assigned_officers"].sum())
deploy = deploy.sort_values("pred_alloc_norm", ascending=False).reset_index(drop=True)

# Adjust by +/-1 while keeping non-negative
i = 0
while diff != 0 and i < 100000:
    idx = i % len(deploy)
    if diff > 0:
        deploy.loc[idx, "assigned_officers"] += 1
        diff -= 1
    else:
        # only subtract if it won't break minimum
        if deploy.loc[idx, "assigned_officers"] > min_per_gn:
            deploy.loc[idx, "assigned_officers"] -= 1
            diff += 1
    i += 1

# Non-deploy set gets 0 officers
non_deploy["pred_alloc_norm"] = 0.0
non_deploy["assigned_officers"] = 0

# Combine results
out = pd.concat([deploy, non_deploy], ignore_index=True)

# Save
out_final = out[[
    "gn_name",
    "risk_score",
    "distance_to_station_km",
    "GN_population",
    "closest_police_station",
    "pred_alloc",
    "assigned_officers"
]].sort_values("assigned_officers", ascending=False)

out_final.to_csv(f"police_allocation_{crime_of_interest}.csv", index=False)
print(f"✅ Saved: police_allocation_{crime_of_interest}.csv")
print(out_final.head(20))
print("Total assigned:", out_final["assigned_officers"].sum())
print("GNs with officers:", (out_final["assigned_officers"] > 0).sum())




✅ Label OK | min/max: 0.0 0.002132146162133812
✅ Model trained | MAE: 2.5538703280238175e-05
✅ Saved: police_allocation_drugs.csv
      gn_name  risk_score  distance_to_station_km  GN_population  \
2   LK2121015    0.960428                    9.52          19665   
10  LK2118005    0.967790                   14.55          12975   
8   LK2121210    0.984335                   11.78          20082   
7   LK2121165    0.953456                    9.05          21394   
6   LK2121170    0.962398                   10.25          22114   
5   LK2124455    0.951041                   11.80          19385   
4   LK2124460    0.958785                   11.92          19901   
0   LK2121050    0.999655                   11.86          19866   
1   LK2121055    0.978979                   11.56          20029   
3   LK2124465    1.000000                   14.07          21821   
9   LK2124450    0.956822                   10.64          18192   
30  LK2115025    0.890429                    8.66     